In [1]:
from config import Config
config = Config()

In [4]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url = config.neo4j_uri,
    username = config.neo4j_username,
    password = config.neo4j_password,
    database = config.neo4j_database
)

In [6]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = config.model, temperature = 0)

In [7]:
from langchain_core.documents import Document


text = """
Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp., formerly Twitter, and his role in the founding of The Boring Company, XAI, Neuralink and OpenAI.
He is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.
Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics and physics. He moved to California in 1995 to attend Stanford University, but dropped out after two days and, with his brother Kimbal, co-founded online city guide software company Zip2.
"""
documents = [Document(page_content = text)]
documents

[Document(metadata={}, page_content="\nElon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp., formerly Twitter, and his role in the founding of The Boring Company, XAI, Neuralink and OpenAI.\nHe is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.\nMusk later transferred to the University of Pennsylvania and received bachelor's degrees in economics and physics. He moved to California in 1995 to attend Stanford University, but dropped out after two days and, with his brother Kimbal, co-founded online city gui

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
graph_transformer = LLMGraphTransformer(llm = llm)
graph_documents = await graph_transformer.aconvert_to_graph_documents(documents)

[GraphDocument(nodes=[Node(id='Elon Reeve Musk', type='Person', properties={}), Node(id='Spacex', type='Company', properties={}), Node(id='Tesla, Inc.', type='Company', properties={}), Node(id='X Corp.', type='Company', properties={}), Node(id='The Boring Company', type='Company', properties={}), Node(id='Xai', type='Company', properties={}), Node(id='Neuralink', type='Company', properties={}), Node(id='Openai', type='Company', properties={}), Node(id='Maye Musk', type='Person', properties={}), Node(id='Errol Musk', type='Person', properties={}), Node(id='University Of Pretoria', type='Educational institution', properties={}), Node(id='Canada', type='Country', properties={}), Node(id="Queen'S University At Kingston", type='Educational institution', properties={}), Node(id='University Of Pennsylvania', type='Educational institution', properties={}), Node(id='Stanford University', type='Educational institution', properties={}), Node(id='Zip2', type='Company', properties={})], relationshi

In [9]:
graph_documents[0].nodes

[Node(id='Elon Reeve Musk', type='Person', properties={}),
 Node(id='Spacex', type='Company', properties={}),
 Node(id='Tesla, Inc.', type='Company', properties={}),
 Node(id='X Corp.', type='Company', properties={}),
 Node(id='The Boring Company', type='Company', properties={}),
 Node(id='Xai', type='Company', properties={}),
 Node(id='Neuralink', type='Company', properties={}),
 Node(id='Openai', type='Company', properties={}),
 Node(id='Maye Musk', type='Person', properties={}),
 Node(id='Errol Musk', type='Person', properties={}),
 Node(id='University Of Pretoria', type='Educational institution', properties={}),
 Node(id='Canada', type='Country', properties={}),
 Node(id="Queen'S University At Kingston", type='Educational institution', properties={}),
 Node(id='University Of Pennsylvania', type='Educational institution', properties={}),
 Node(id='Stanford University', type='Educational institution', properties={}),
 Node(id='Zip2', type='Company', properties={})]

In [11]:
graph_documents[0].relationships

[Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Spacex', type='Company', properties={}), type='FOUNDED', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Tesla, Inc.', type='Company', properties={}), type='FOUNDED', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='X Corp.', type='Company', properties={}), type='OWNED', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='The Boring Company', type='Company', properties={}), type='FOUNDED', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Xai', type='Company', properties={}), type='FOUNDED', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Neuralink', type='Company', properties={}), type='FO

In [12]:
movie_query= """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE (m: Movie{id: row. movieId})
SET m.released = date(row. released),
m.title = row.title,
m. imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, 'I') |
MERGE (p: Person {name: trim(director)})
MERGE (P) -[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
MERGE (p:Person {name:trim(actor)})
MERGE (p) -[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, 'l') |
MERGE (g:Genre {name:trim(genre)})
MERGE (m) - [:IN_GENRE] ->(g))
"""

In [13]:
graph.query(movie_query)

[]